# ノンパラメトリック手法（k-NN）による異常検知

ここまで紹介した手法は、データが正規分布や混合正規分布といった特定の分布に従うことを前提としていました。しかし、実際のデータがこれらの分布に適合していなければ、誤報や見逃しが増える原因となります。可視化やEDAの結果から正規分布や混合正規分布に従わないと判断できる場合は、これらの分布に依存しない異常検知手法を検討すべきです。

一方で多変数においては一般的に可視化やEDAの結果から確率分布の種類を特定することが困難であるため、特定の種類の確率分布をモデリングに用いることなく柔軟にデータに適合できる、**ノンパラメトリック手法**（non-parametric methods）がよく用いられます。ノンパラメトリック手法は分布形状に強い仮定を置かず、限られた少数のパラメータでは表現できないモデル全般を指し、異常検知においては以下のノンパラメトリック手法がよく用いられます。

- **k-NN**（k-nearest neighbors：k近傍法）：推論対象のデータ$x$から$k$番目に近い学習データまでの距離に基づき、判定を行う手法
- KDE（kernel density estimation：カーネル密度推定）：データ1個ごとにカーネル関数に基づき確率密度関数を割り当てる手法
- LoF（local outlier factor：局所外れ値因子法）：k-NNを発展させ、k番目よりも近い点すべての距離を考慮して密度に基づく指標を求める手法
- OC-SVM（one-class support vector machines：1クラスサポートベクターマシン）：SVMを拡張し、正常データからの外れ度合いに基づいて決定境界を構築する手法

ここではこれらの手法の中でも最もシンプルで広く用いられているk-NNを用いた異常検知を、Pythonを用いて以下の手順で実装する方法を解説します。

- A. 変数の選択
- B. モデルの学習
- C. 推論

今回は対象データとして、2クラスタのGMMに基づき生成したサンプルデータ（GMMによる異常検知の実装で用いたデータと同じもの）を使用します。

## A. 変数の選択

多変数のホテリング理論（[ch6_3_Multivariate_Hotelling.ipynb](https://github.com/ghmagazine/python_anomaly_detection_book/blob/main/notebooks/ch6_3_Multivariate_Hotelling.ipynb)）と同様のため、省略します。

## B. モデルの学習

学習フェーズでは、学習データの標準化（k-NNは特徴空間上での距離に基づくアルゴリズムのため、前処理として標準化が必要）と保持、および異常度の分位点に基づくしきい値の算出を行います。Pythonでは、scikit-learnの[sklearn.neighbors.NearestNeighbors](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html)クラスを用いると、k-NNを簡単に実装できます。

In [ ]:
# コード6.9 k-NN よる異常検知の実装例（学習）
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

###### 学習データの読み込みと前処理######
SEED=42 # 乱数シードを指定（結果の再現性を担保）
rng = np.random.default_rng(seed=SEED)
# 学習用サンプルとして2パターンの2次元正規分布から300個ずつデータ生成
MU_1 = [0, 0]
SIGMA_1 = [[1, 0],
           [0, 1]]
X_train_1 = rng.multivariate_normal(MU_1, SIGMA_1, 600)
MU_2 = [4, 0]
SIGMA_2 = [[2, 1],
           [1, 1]]
X_train_2 = rng.multivariate_normal(MU_2, SIGMA_2, 600)
X_train = np.concatenate([X_train_1, X_train_2])
# しきい値計算用データを学習データから分離
X_for_train, X_for_threshold = train_test_split(X_train, train_size=0.5, random_state=SEED)

###### 学習ステップ1. 正常のモデルを作成する######
# k-NNモデルを作成
knn = NearestNeighbors(n_neighbors=5, algorithm='auto', metric='minkowski')
# 標準化とパイプライン化する
pipe = Pipeline([("scaler", StandardScaler()), ("knn", knn)])
pipe.fit(X_for_train) # モデルを学習

###### 学習ステップ2. 異常を表す指標（異常度）を定義する######
# 式を定義するのみでプログラム上は処理を実施しない

###### 学習ステップ3. 異常度にしきい値を設ける######
TARGET_FP_RATE = 0.0027 # ターゲットとする誤報率（正規分布の3σ相当=0.0027）
X_threshold_std = pipe.named_steps[ # 標準化
    'scaler'].transform(X_for_threshold)
distances, indices = pipe.named_steps[ # しきい値計算用データにk-NN適用
    'knn'].kneighbors(X_threshold_std)
thresh_anom_scores = distances[:, -1] # 異常度εを算出
# 異常度の分位点からしきい値の算出
a_th = np.quantile(thresh_anom_scores, 1-TARGET_FP_RATE)

###### 求めたしきい値を表示######
print(f'a_th={a_th}')

k-NNモデルと前処理の標準化を一体化して処理するために、scikit-learnのパイプライン機能（`sklearn.pipeline.Pipeline`クラス）を使用しています。

また`sklearn.neighbors.NearestNeighbors`クラスの`n_neighbors`引数でk-NNのパラメータ$k$を、`algorithm`引数で近傍点探索のアルゴリズム、`metric`で距離の計算方法を指定できます。`algorithm`および`metric`引数の詳細は、[scikit-learnのドキュメント](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html)を参照してください。最も一般的なユークリッド距離を用いる場合は、`metric="minkowski"`を指定すると良いでしょう。

学習したモデル（k-NNと標準化のパイプラインモデル）を保存して推論に活用するためには、3章で紹介したpickleなどを使用することが一般的です（ここでは簡単のため、学習したモデルの`pipe`インスタンスをそのまま推論に用いることとします）。

## C. 推論

学習フェーズで得られた学習済モデルとしきい値を用いて、推論データに対する異常度の算出と異常判定を行います（推論データもGMMと同じものを使用します）。

In [ ]:
# コード6.10 k-NN よる異常検知の実装例（推論）
import pandas as pd

A_TH=0.8908177244386846 # 異常度のしきい値

###### 推論データの読み込みと前処理######
# 推論用の正常データとして二つの正規分布からデータを200個ずつ生成
rng = np.random.default_rng(seed=42)
X_inference_norm_1 = rng.multivariate_normal([0, 0], [[1, 0], [0, 1]], 100)
X_inference_norm_2 = rng.multivariate_normal([4, 0], [[2, 1], [1, 1]], 100)
X_inference_norm = np.concatenate([X_inference_norm_1, X_inference_norm_2])
df_inference_norm = pd.DataFrame(X_inference_norm, columns=["x1", "x2"])
df_inference_norm['label'] = 'normal'
# 推論用の異常データとして平均[3, 2]、2の正規分布からデータを20個生成
X_inference_anom = rng.multivariate_normal([3, 2], [[1, 0], [0, 1]], 20)
df_inference_anom = pd.DataFrame(X_inference_anom, columns=["x1", "x2"])
df_inference_anom['label'] = 'anomaly'
# 推論用の正常データと異常データを合体
df_inference = pd.concat([df_inference_norm, df_inference_anom], axis=0)
df_inference = df_inference.reset_index(drop=True)
X_inference = df_inference[["x1", "x2"]].to_numpy()

###### 推論を実行######
# 異常度を算出
X_inference_std = pipe.named_steps[ # 標準化
    'scaler'].transform(X_inference)
distances, indices = pipe.named_steps[ # 推論データにk-NN適用
    'knn'].kneighbors(X_inference_std)
anomaly_scores = distances[:, -1] # 異常度εを算出
# しきい値により異常の有無を判定
pred = np.where(anomaly_scores > A_TH, 'anomaly', 'normal')
# 推論結果を表示
print(pred)

推論結果の決定境界（異常と正常の判定の境界）を、$k=1,5,25$の3パターンについて可視化してみます。

In [ ]:
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

K_LIST = [1, 5, 25]

###### 学習データの読込と前処理 ######
SEED=42  # 乱数シードを指定（結果の再現性を担保）
rng = np.random.default_rng(seed=SEED)
# 学習用サンプルとして2パターンの2次元正規分布から300個ずつデータ生成
MU_1 = [0, 0]
SIGMA_1 = [[1, 0],
           [0, 1]]
X_train_1 = rng.multivariate_normal(MU_1, SIGMA_1, 600)
MU_2 = [4, 0]
SIGMA_2 = [[2, 1],
           [1, 1]]
X_train_2 = rng.multivariate_normal(MU_2, SIGMA_2, 600)
X_train = np.concatenate([X_train_1, X_train_2])
# しきい値計算用データを学習データから分離
X_for_train, X_for_threshold = train_test_split(X_train, train_size=0.5, random_state=SEED)

# 描画用のFigureとAxesを生成
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 5))

for i, k in enumerate(K_LIST):
    ###### 学習ステップ1. 正常のモデルを作成する ######
    knn = NearestNeighbors(n_neighbors=k, algorithm='auto', metric='minkowski')  # k-NNモデルを作成
    pipe = Pipeline([("scaler", StandardScaler()), ("knn", knn)])  # 標準化とパイプライン化する
    pipe.fit(X_for_train)  # モデルを学習

    ###### 学習ステップ2. 異常を表す指標（異常度）を定義する ######
    # 式を定義するのみでプログラム上は処理を実施しない

    ###### 学習ステップ3. 異常度にしきい値を設ける ######
    TARGET_FP_RATE = 0.0027  # ターゲットとする誤報率（正規分布の3σ相当=0.0027）
    X_threshold_std = pipe.named_steps[  # 標準化
        'scaler'].transform(X_for_threshold)
    distances, indices = pipe.named_steps[  # しきい値計算用データにk-NN適用
        'knn'].kneighbors(X_threshold_std)
    thresh_anom_scores = distances[:, -1]  # 異常度εを算出
    a_th = np.quantile(thresh_anom_scores, 1 - TARGET_FP_RATE)  # 異常度の分位点からしきい値算出

    ############ 推論 ############
    ###### 推論データの読込と前処理 ######
    # 正常データとして2つの正規分布からデータを200個ずつ生成
    rng = np.random.default_rng(seed=42)
    X_inference_norm_1 = rng.multivariate_normal([0, 0], [[1, 0], [0, 1]], 100)
    X_inference_norm_2 = rng.multivariate_normal([4, 0], [[2, 1], [1, 1]], 100)
    X_inference_norm = np.concatenate([X_inference_norm_1, X_inference_norm_2])
    df_inference_norm = pd.DataFrame(X_inference_norm, columns=["x1", "x2"])
    df_inference_norm['label'] = 'normal'
    # 異常データとして平均[3, 2]、2の正規分布からデータを20個生成
    X_inference_anom = rng.multivariate_normal([3, 2], [[1, 0], [0, 1]], 20)
    df_inference_anom = pd.DataFrame(X_inference_anom, columns=["x1", "x2"])
    df_inference_anom['label'] = 'anomaly'
    # 正常データと異常データを合体
    df_inference = pd.concat([df_inference_norm, df_inference_anom], axis=0)
    df_inference = df_inference.reset_index(drop=True)
    X_inference = df_inference[["x1", "x2"]].to_numpy()

    ###### 推論を実行 ######
    # 異常度を算出
    X_inference_std = pipe.named_steps[  # 標準化
        'scaler'].transform(X_inference)
    distances, indices = pipe.named_steps[  # 推論データにk-NN適用
        'knn'].kneighbors(X_inference_std)
    anomaly_scores = distances[:, -1]  # 異常度εを算出
    # しきい値により異常の有無を判定
    pred = np.where(anomaly_scores > a_th, 'anomaly', 'normal')

    ############ 正常と異常の境界をプロット ############
    ###### 正常と異常の範囲を色分け ######
    # (x1,x2)格子点を作成（'temp2'をx1としていることに注意）
    x1_grid = np.linspace(df_inference['x1'].min()-2,
                        df_inference['x1'].max()+2, 300)
    x2_grid = np.linspace(df_inference['x2'].min()-1,
                        df_inference['x2'].max()+1, 300)
    X1, X2 = np.meshgrid(x1_grid, x2_grid)
    X_grid = np.c_[X1.ravel(), X2.ravel()]
    # 異常度を算出
    X_grid_std = pipe.named_steps[  # 標準化
        'scaler'].transform(X_grid)
    distances, indices = pipe.named_steps[  # 格子点データにk-NN適用
        'knn'].kneighbors(X_grid_std)
    anomaly_scores_grid = distances[:, -1]  # 異常度εを算出
    # しきい値判定
    pred_grid = np.where(anomaly_scores_grid > a_th, 0, 1)
    # 正常と異常の境界をプロット
    pred_pivot = pred_grid.reshape(X1.shape)
    axes[i].contourf(X1, X2, pred_pivot,
                cmap=cm.gray, alpha=0.5)

    ###### 各データを散布図としてプロット ######
    sns.scatterplot(data=df_inference, x='x1', y='x2',
        hue='label', palette=['#999999', '#111111'],
        ax=axes[i]
    )
    # 凡例を追加
    axes[i].legend()
    axes[i].set_title(f'k={k}', fontsize=16)

# グラフを表示
plt.show()

$k$が⼩さいと学習データ一つ一つの影響が⼤きい複雑な形状の境界となり、$k$が⼤きいと平滑化された滑らかな境界となり、学習データ一つ一つの影響が⼩さくなることがわかります。$k$を変えたときの変化は、ちょうど3章のSVMにおいて$\gamma$を変えたときの変化と似ています。$k$が⼩さいと個々の学習データに左右されやすい過学習寄りの設定に$k$が⼤きいと未学習寄りの設定となります。適切な$k$の値は、9章で紹介する交差検証による性能を最⼤化するようハイパーパラメータチューニングにより決めることが一般的です。